<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/02_data_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Purpose: Inspect the actual datasets before annotation validation, conversion and training.

This notebook doesn't modify Raw Data. It reports image count, annotation formats, image integrity, dimensions, sample images and te observed RDD2022 YOLO Labels

##Target Classes


1.   Crack
2.   Pothole
3.   Surface deterioration



In [ ]:
from pathlib import Path
from collections import Counter
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"

RAW_DIR = DATA_DIR / "Raw Data"
SELECTED_DIR = DATA_DIR / "Selected Data"
ANNOTATED_DIR = DATA_DIR / "Annotated Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

SDNET_DIR = RAW_DIR / "SDNET2018"
RDD_DIR = RAW_DIR / "RDD2022"
KENYAN_DIR = RAW_DIR / "Kenyan_Data"

SDNET_ANNOTATED = ANNOTATED_DIR / "SDNET2018"
RDD_ANNOTATED = ANNOTATED_DIR / "RDD2022"
KENYAN_ANNOTATED = ANNOTATED_DIR / "Kenyan_Data"

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR.exists())
print("Selected data:", SELECTED_DIR.exists())
print("Annotated data:", ANNOTATED_DIR.exists())
print("Processed data:", PROCESSED_DIR.exists())

## 1. Dataset Overview

The counts below are taken from the directories currently present in the Google Drive. They are not hard coded

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG", ".WEBP"}
ANNOTATION_EXTENSIONS = {".xml", ".txt", ".json"}

def files_with_extensions(directory, extensions):
    directory = Path(directory)
    if not directory.exists():
        return []
    return [p for p in directory.rglob("*") if p.is_file() and p.suffix in extensions]

def count_images(directory):
    return len(files_with_extensions(directory, IMAGE_EXTENSIONS))

def count_files(directory, extensions):
    return len(files_with_extensions(directory, extensions))

datasets = {
    "SDNET2018": SDNET_DIR,
    "RDD2022": RDD_DIR,
    "Kenyan_Data": KENYAN_DIR,
}

overview = []
for name, path in datasets.items():
    overview.append({
        "dataset": name,
        "images": count_images(path),
        "xml": count_files(path, {".xml"}),
        "yolo_txt": count_files(path, {".txt"}),
        "json": count_files(path, {".json"}),
    })

overview_df = pd.DataFrame(overview)
display(overview_df)

## 2. Image Integrity Checks

A readable image is required before it can enter annotation validation or training

In [ ]:
def check_image_integrity(directory, max_images=None):
    image_paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    if max_images is not None:
        image_paths = image_paths[:max_images]

    valid = 0
    invalid = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                img.verify()
            valid += 1
        except Exception as exc:
            invalid.append((str(path), str(exc)))

    return len(image_paths), valid, invalid

integrity_rows = []
for name, path in datasets.items():
    total, valid, invalid = check_image_integrity(path)
    integrity_rows.append({
        "dataset": name,
        "checked": total,
        "valid": valid,
        "invalid": len(invalid)
    })

integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df)

In [ ]:
# Show a few corrupted/unreadable files if any were found.
for name, path in datasets.items():
    _, _, invalid = check_image_integrity(path)
    if invalid:
        print(f"\n{name} invalid examples:")
        for item in invalid[:10]:
            print(item)

## 3. Image Dimensions

Dimension statistics help identify unusual files and dataset difference. They are descriptive only: no resizing is performed here.

In [ ]:
def image_dimension_summary(directory, max_images=None):
    image_paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    if max_images is not None:
        image_paths = image_paths[:max_images]

    rows = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                w, h = img.size
            rows.append({"path": str(path), "width": w, "height": h, "aspect_ratio": round(w / h, 4)})
        except Exception:
            pass

    return pd.DataFrame(rows)

dimension_summaries = {}

for name, path in datasets.items():
    df = image_dimension_summary(path)
    dimension_summaries[name] = df

    if not df.empty:
        print(f"\n{name}")
        print("Images measured:", len(df))
        print("Width:", df["width"].min(), "to", df["width"].max())
        print("Height:", df["height"].min(), "to", df["height"].max())
        print("Mean aspect ratio:", round(df["aspect_ratio"].mean(), 3))

## 4. Random Image Visualization

In [ ]:
def random_image_paths(directory, n=4, seed=42):
    paths = files_with_extensions(directory, IMAGE_EXTENSIONS)
    rng = random.Random(seed)
    return rng.sample(paths, min(n, len(paths)))

for name, path in datasets.items():
    samples = random_image_paths(path, n=3, seed=42)

    if not samples:
        continue

    fig, axes = plt.subplots(1, len(samples), figsize=(15, 5))
    if len(samples) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, samples):
        try:
            with Image.open(img_path) as img:
                ax.imshow(img.convert("RGB"))
            ax.set_title(img_path.name, fontsize=8)
            ax.axis("off")
        except Exception as exc:
            ax.set_title(f"Unreadable\n{exc}")
            ax.axis("off")

    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

## 5. RDD2022 YOLO Annotation Inspection

Notebook 01 established that the current RDD2022 copy contains YOLO .txt files and no XML files.

This section inspects the actual class IDs and annotation syntax. Do not finalize the class mapping until the output below has been checked.

In [ ]:
RDD_TXT_FILES = files_with_extensions(RDD_DIR, {".txt"})

def read_yolo_lines(txt_path):
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return [line.strip() for line in f if line.strip()]

class_id_counts = Counter()
invalid_line_examples = []

for txt_path in RDD_TXT_FILES:
    for line in read_yolo_lines(txt_path):
        parts = line.split()
        if len(parts) != 5:
            if len(invalid_line_examples) < 20:
                invalid_line_examples.append((str(txt_path), line))
            continue

        try:
            class_id = int(parts[0])
            values = [float(v) for v in parts[1:]]
            class_id_counts[class_id] += 1
        except ValueError:
            if len(invalid_line_examples) < 20:
                invalid_line_examples.append((str(txt_path), line))

print("Observed RDD2022 class IDs:")
print(dict(sorted(class_id_counts.items())))

if invalid_line_examples:
    print("\nInvalid YOLO line examples:")
    for item in invalid_line_examples[:20]:
        print(item)

In [ ]:
# Show actual sample YOLO annotation files.
for txt_path in RDD_TXT_FILES[:5]:
    print("\nFILE:", txt_path)
    for line in read_yolo_lines(txt_path)[:10]:
        print(line)

## RDD mapping checkpoint

For standard RDD four-class road-damage taxonomy, the expected mapping is:

*   D00 - Crack
*   D10 - Crack
*   D20 - Crack
*   D40 - Pothole

If the obsevered class IDs are not the expected four IDs, STOP before Notebook 03 and update the mapping from the actual annotation files.

Do not use D43 or D44 as RDD2022 classes unless your actual downloaded dataset demonstrably contains those classes. They are part of the four-class RDD2022 mappinf being used here





## 6. Basic Image-Annotation Pairing Check for RDD2022

In [ ]:
def build_stem_index(paths):
    index = {}
    for p in paths:
        index.setdefault(p.stem, []).append(p)
    return index

rdd_images = files_with_extensions(RDD_DIR, IMAGE_EXTENSIONS)
rdd_labels = files_with_extensions(RDD_DIR, {".txt"})

image_index = build_stem_index(rdd_images)
label_index = build_stem_index(rdd_labels)

paired = 0
missing_images = []
duplicate_image_stems = []

for stem, labels in label_index.items():
    images = image_index.get(stem, [])
    if len(images) == 1:
        paired += 1
    elif len(images) == 0:
        missing_images.append(stem)
    else:
        duplicate_image_stems.append((stem, len(images)))

print("YOLO annotation files:", len(rdd_labels))
print("Unique label stems:", len(label_index))
print("Paired unique image stems:", paired)
print("Missing image examples:", missing_images[:10])
print("Duplicate image stem examples:", duplicate_image_stems[:10])

## 7. Dataset Inspection Summary

In [ ]:
print("DATASET SUMMARY")
print("=" * 60)
display(overview_df)

print("\nNEXT STAGE")
print("Notebook 03 will validate annotations, apply the target class mapping,")
print("construct a normalized manifest, and create leakage-aware dataset splits.")